# Lab 05: Extract economic data from a public API

You will request real World Bank data, explore the JSON it returns, turn it into a table, wrap it in a defensive function with a cache fallback, and save the result with metadata.

For each step:
1. Read the explanation. The **Example** shows the pattern to follow; look back at the matching slide or demo notebook if you need more.
2. Replace each `...` or `# TODO` with your code and run the cell (**Shift+Enter**).
3. Run the **check** cell below it: it prints `OK` when you are right, or shows an `AssertionError` if not.

Steps build on each other, so work in order. After a restart use *Run > Run All Above Selected Cell*.

**Data source:** the World Bank Indicators API (free, no key). If the internet is unavailable, cached responses are in `data/cache/`; steps that need the network tell you what to run instead.

---
# Part A: JSON

## A1. Set up
Imports, the API address and the 14 course economies.

*Run this cell; no changes needed.*

In [ ]:
import json
from pathlib import Path
import requests
import pandas as pd

BASE = "https://api.worldbank.org/v2"
CACHE = Path("../../data/cache")
OUT = Path("output")
OUT.mkdir(exist_ok=True)
COUNTRIES = ["AUS", "BRA", "CAN", "CHN", "DEU", "IND", "JPN",
             "KEN", "MEX", "NGA", "GBR", "USA", "VNM", "ZAF"]

## A2. JSON text to Python
`json.loads(text)` turns JSON text into Python dictionaries and lists. Load `text` into `data`, then store the ISO3 code in `iso3` and the **second** value in the `gdp` list in `gdp_2`.

**Example**
```python
d = json.loads('{"a": [10, 20]}')
d["a"][0]   # 10
```

In [ ]:
text = '{"iso3": "KEN", "gdp": [104.9, 107.4]}'
data = ...    # TODO
iso3 = ...    # TODO
gdp_2 = ...   # TODO
print(iso3, gdp_2)

In [ ]:
# check
assert iso3 == "KEN" and gdp_2 == 107.4
print("A2 OK")

## A3. Nested JSON
This is one real World Bank record. Some values are dictionaries **inside** the dictionary. Get the country name (inside `"country"`), the year (`"date"`) and the value.

**Example**
```python
record["indicator"]["id"]   # 'NE.EXP.GNFS.ZS'
```

In [ ]:
record = json.loads('''{"indicator": {"id": "NE.EXP.GNFS.ZS", "value": "Exports of goods and services (% of GDP)"},
 "country": {"id": "KE", "value": "Kenya"},
 "countryiso3code": "KEN", "date": "2023", "value": 16.7}''')
name = ...    # TODO
year = ...    # TODO
value = ...   # TODO
print(name, year, value)

In [ ]:
# check
assert name == "Kenya" and year == "2023" and value == 16.7
print("A3 OK")

---
# Part B: Your first API request

## B1. Send the request
Complete the request for Kenya's exports as % of GDP (`NE.EXP.GNFS.ZS`), 2015 to 2023. `params` becomes the `?format=json&date=...` part of the URL.

**Example**
```python
resp = requests.get(url, params={"format": "json"}, timeout=30)
```

In [ ]:
url = f"{BASE}/country/KEN/indicator/NE.EXP.GNFS.ZS"
params = {"format": "json", "date": "2015:2023"}
resp = ...   # TODO: requests.get with params and timeout=30
print(resp.status_code)
print(resp.url)

In [ ]:
# check
assert resp.status_code == 200
print("B1 OK")

## B2. Unpack the response
`resp.json()` returns a list of two things: metadata, then the records. Unpack it: `meta, records = ...`. Print the metadata and how many records came back.

*No internet?* Instead run: `records = [r for r in json.loads((CACHE / "wb_exports_pct_gdp.json").read_text())[1] if r["countryiso3code"] == "KEN"]`

In [ ]:
meta, records = ...   # TODO
print(meta)
print(len(records), "records")

In [ ]:
# check
assert len(records) == 9
print("B2 OK")

---
# Part C: From records to a table

## C1. Loop over the records
Print the year and value of each record (as in A3).

In [ ]:
for r in records:
    # TODO
    pass

## C2. Count missing values
Some years are not yet published: their value is `None`. Count them into `missing` (test with `r["value"] is None`).

In [ ]:
missing = 0
for r in records:
    # TODO
    pass
print(missing, "missing")

In [ ]:
# check
assert missing == sum(r["value"] is None for r in records)
print("C2 OK")

## C3. Build tidy rows
Build `rows`: one dictionary per record with keys `iso3`, `year` (converted to `int`) and `exports_pct_gdp`.

In [ ]:
rows = []
for r in records:
    rows.append({
        "iso3": ...,              # TODO
        "year": ...,              # TODO
        "exports_pct_gdp": ...,   # TODO
    })
print(rows[0])

In [ ]:
# check
assert len(rows) == 9 and isinstance(rows[0]["year"], int)
print("C3 OK")

## C4. Into pandas
`pd.DataFrame(rows)` turns a list of dictionaries into a table. Sort it by year.

In [ ]:
kenya = ...   # TODO
kenya

In [ ]:
# check
assert list(kenya.columns) == ["iso3", "year", "exports_pct_gdp"]
print("C4 OK")

---
# Part D: A reusable function

## D1. First version
Turn Part B into a function. `countries` is a list of ISO3 codes joined with `;` (e.g. `"KEN;NGA"`). Set `per_page` to 1000 so all records come back in one page. Return `resp.json()[1]`.

In [ ]:
def fetch_indicator(countries, indicator, start, end):
    codes = ";".join(countries)
    url = f"{BASE}/country/{codes}/indicator/{indicator}"
    params = {"format": "json", "date": f"{start}:{end}", "per_page": 1000}
    # TODO: send the request (timeout=30) and return the records

recs = fetch_indicator(["KEN", "NGA"], "NY.GDP.MKTP.CD", 2020, 2023)
print(len(recs))

In [ ]:
# check
assert len(recs) == 8
print("D1 OK")

## D2. What happens with a bad code?
An invalid country code still returns status **200**, but the body is an error message. Our first version would crash or return nonsense.

*Run this cell; no changes needed.*

In [ ]:
bad = requests.get(f"{BASE}/country/XXX/indicator/NY.GDP.MKTP.CD", params={"format": "json"}, timeout=30)
print(bad.status_code)
print(bad.json())

## D3. Make it defensive
Improve `fetch_indicator` (copy your D1 version):
1. call `resp.raise_for_status()` after the request
2. store `payload = resp.json()`
3. if `len(payload) < 2 or not payload[1]`, `raise ValueError(...)`
4. otherwise return `payload[1]`

In [ ]:
def fetch_indicator(countries, indicator, start, end):
    # TODO: copy D1 and add the checks
    pass

In [ ]:
# check
assert len(fetch_indicator(["KEN", "NGA"], "NY.GDP.MKTP.CD", 2020, 2023)) == 8
try:
    fetch_indicator(["XXX"], "NY.GDP.MKTP.CD", 2020, 2023)
    raise AssertionError("expected a ValueError")
except ValueError:
    pass
print("D3 OK")

## D4. Records to a DataFrame
Wrap Part C in a function `to_frame(records, name)` that returns a DataFrame with columns `iso3`, `country`, `year` and a value column called `name`.

In [ ]:
def to_frame(records, name):
    rows = []
    for r in records:
        # TODO: append a dictionary with iso3, country, year and name
        pass
    return pd.DataFrame(rows)

In [ ]:
# check
t = to_frame(records, "x")
assert list(t.columns) == ["iso3", "country", "year", "x"]
print("D4 OK")

---
# Part E: Two indicators, with a fallback

## E1. Loading a cached response
If the API fails we use a saved copy. This function reads a cached file and returns its records.

*Run this cell; no changes needed.*

In [ ]:
def load_cached(filename):
    text = (CACHE / filename).read_text(encoding="utf-8")
    return json.loads(text)[1]

print(len(load_cached("wb_gdp_usd.json")), "cached records")

## E2. Try the API, fall back to the cache
For each indicator, **try** `fetch_indicator` for all 14 `COUNTRIES`, 2015 to 2023. If a `requests.RequestException` occurs, print a message and use `load_cached(cache_file)`. Append `to_frame(recs, name)` to `frames`.

In [ ]:
INDICATORS = {
    "exports_pct_gdp": ("NE.EXP.GNFS.ZS", "wb_exports_pct_gdp.json"),
    "gdp_usd": ("NY.GDP.MKTP.CD", "wb_gdp_usd.json"),
}
frames = []
for name, (code, cache_file) in INDICATORS.items():
    try:
        recs = ...   # TODO
    except requests.RequestException as e:
        print("API failed, using cache:", e)
        recs = ...   # TODO
    frames.append(to_frame(recs, name))
print([len(f) for f in frames])

In [ ]:
# check
assert [len(f) for f in frames] == [126, 126]
print("E2 OK")

## E3. Merge the two indicators
Join the two frames side by side on `iso3`, `country` and `year`.

**Example**
```python
a.merge(b, on=["iso3", "year"])
```

In [ ]:
wb = ...   # TODO
wb.head()

In [ ]:
# check
assert {"exports_pct_gdp", "gdp_usd"} <= set(wb.columns) and wb["iso3"].nunique() == 14
print("E3 OK")

---
# Part F: Save the result

## F1. Tidy up
Add `gdp_usd_bn` (GDP divided by 1e9, rounded to 1 dp). Print how many rows have a missing value in either indicator (`wb.isna().any(axis=1).sum()`).

In [ ]:
# TODO

In [ ]:
# check
assert "gdp_usd_bn" in wb.columns
print("F1 OK")

## F2. Save with metadata
Save `wb` to `output/wb_indicators.csv` (no index). The metadata file records where the data came from; complete the row count.

In [ ]:
from datetime import date
# TODO: save the CSV
metadata = {
    "source": BASE,
    "indicators": {k: v[0] for k, v in INDICATORS.items()},
    "retrieved": date.today().isoformat(),
    "rows": ...,   # TODO
}
(OUT / "wb_indicators_metadata.json").write_text(json.dumps(metadata, indent=2))
print(metadata)

In [ ]:
# check
assert (OUT / "wb_indicators.csv").exists()
assert json.loads((OUT / "wb_indicators_metadata.json").read_text())["rows"] == 126
print("F2 OK")

**Question:** Which country had the highest exports as a share of GDP in the latest year with data? (Try `wb.sort_values("exports_pct_gdp", ascending=False).head()`.)

*Your answer:* (double-click to edit)